<a href="https://colab.research.google.com/github/u202021380/lesiones-cutaneas/blob/main/notebooks/entrenamiento_HU06_HU16.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Entrenamiento y validación del modelo (HU06 y HU16)

**Sprints 1 y 2.** Entrena MobileNetV2 (arquitectura del producto, decisión D1) y MobileNetV3-Large (referencia de comparación, HU16) con PAD-UFES-20, con el mismo protocolo:

- partición por paciente 70/15/15 (CP06);
- Transfer Learning en dos fases, con aumento de datos y pesos por clase;
- métricas en la prueba interna: exactitud, exactitud balanceada, AUC-ROC con IC 95 %, sensibilidad y especificidad por clase, matriz de confusión y AUC-ROC binario maligno frente a no maligno (para comparar después con DDI);
- tiempo de inferencia en CPU y tamaño del modelo;
- análisis de confianza para el umbral de HU12 y calibración del control de calidad de HU09.

**Antes de empezar:** Entorno de ejecución → Cambiar tipo de entorno de ejecución → **GPU (T4)**.
Al final se descarga `resultados_entrenamiento.zip` con el modelo, las métricas y las figuras. **No edites los resultados a mano**: son la evidencia del Sprint.

In [1]:
!nvidia-smi -L
import sys, platform, tensorflow as tf, keras
print("Python", platform.python_version(), "| TensorFlow", tf.__version__, "| Keras", keras.__version__)
print("GPU:", tf.config.list_physical_devices("GPU"))

GPU 0: Tesla T4 (UUID: GPU-3b3bd479-d9bf-076e-b435-e3f00fdad6d0)
Python 3.13.16 | TensorFlow 2.21.0 | Keras 3.13.2
GPU: [PhysicalDevice(name='/physical_device:GPU:0', device_type='GPU')]


## 1. Código del proyecto
Opción A: clona tu repositorio de GitHub (cambia la URL). Opción B: sube el archivo `lesiones-cutaneas.zip` al panel de archivos y deja `REPO_URL = ""`.

In [2]:
REPO_URL = "https://github.com/u202021380/lesiones-cutaneas.git"
import os
if REPO_URL:
    !git clone -q {REPO_URL} /content/lesiones-cutaneas
elif not os.path.exists("/content/lesiones-cutaneas"):
    !unzip -q /content/lesiones-cutaneas.zip -d /content/
%cd /content/lesiones-cutaneas
sys.path.insert(0, "/content/lesiones-cutaneas")
!ls

/content/lesiones-cutaneas
app	    models     pytest.ini  requirements-dev.txt  resultados  tests
Dockerfile  notebooks  README.md   requirements.txt	 scripts     training


## 2. Descarga de PAD-UFES-20
Fuente: Pacheco et al. (2020), Mendeley Data, doi:10.17632/zr7vgbcyr2.1 (licencia CC BY 4.0).
Si la descarga directa falla, descarga el dataset desde https://data.mendeley.com/datasets/zr7vgbcyr2/1, súbelo a tu Google Drive y cambia `RUTA_DRIVE`.

In [ ]:
DATOS = "/content/pad-ufes-20"
RUTA_DRIVE = "/content/drive/MyDrive/zr7vgbcyr2-1"
os.makedirs(DATOS, exist_ok=True)
if RUTA_DRIVE:
    from google.colab import drive
    drive.mount("/content/drive")
    !cp "{RUTA_DRIVE}" {DATOS}/
else:
    !wget -q -O {DATOS}/pad-ufes-20.zip "https://prod-dcd-datasets-cache-zipfiles.s3.eu-west-1.amazonaws.com/zr7vgbcyr2-1.zip"
from training.datos import descomprimir_todo, cargar_metadatos
descomprimir_todo(DATOS)
df = cargar_metadatos(DATOS)
print(len(df), "imágenes de", df["patient_id"].nunique(), "pacientes")
print(df["diagnostic"].value_counts())

Mounted at /content/drive
cp: cannot stat '/content/drive/MyDrive/zr7vgbcyr2-1': No such file or directory


## 3. Partición por paciente (HU06, CP06)

In [ ]:
from training.datos import particionar_por_paciente, verificar_sin_fuga
import json
os.makedirs("resultados", exist_ok=True)
df = particionar_por_paciente(df, semilla=42)
cp06 = verificar_sin_fuga(df)
print(json.dumps(cp06, indent=2))
assert cp06["sin_fuga"], "Hay pacientes en dos conjuntos"
df[["patient_id", "lesion_id", "img_id", "diagnostic", "fitspatrick", "conjunto"]].to_csv("resultados/particion.csv", index=False)
print(df.groupby(["conjunto", "diagnostic"]).size().unstack(fill_value=0))

## 4. Carga de imágenes (misma función de preprocesamiento que la API, HU02)

In [ ]:
from training.datos import cargar_imagenes, crear_dataset
import numpy as np
partes = {c: df[df["conjunto"] == c].reset_index(drop=True) for c in ("train", "val", "test")}
X = {c: cargar_imagenes(p["ruta"].tolist()) for c, p in partes.items()}
Y = {c: p["etiqueta"].to_numpy() for c, p in partes.items()}
ds_train = crear_dataset(X["train"], Y["train"], aumentar=True)
ds_val = crear_dataset(X["val"], Y["val"])
ds_test = crear_dataset(X["test"], Y["test"])
print({c: x.shape for c, x in X.items()})

## 5. MobileNetV2: entrenamiento y evaluación (HU06)

In [ ]:
from training.entrenar import entrenar
from training.evaluacion import metricas, analisis_confianza, tiempo_inferencia_cpu
resultados = {}

def evaluar(modelo, nombre, ruta_modelo, historial):
    p_val = modelo.predict(ds_val, verbose=0)
    p_test = modelo.predict(ds_test, verbose=0)
    r = {
        "arquitectura": nombre,
        "versiones": {"tensorflow": tf.__version__, "keras": keras.__version__},
        "particion": cp06,
        "validacion": metricas(Y["val"], p_val),
        "prueba": metricas(Y["test"], p_test),
        "confianza_validacion": analisis_confianza(Y["val"], p_val),
        "confianza_prueba": analisis_confianza(Y["test"], p_test),
        "inferencia": tiempo_inferencia_cpu(ruta_modelo),
        "minutos_entrenamiento": historial["minutos"],
    }
    with open(f"resultados/metricas_{nombre}.json", "w", encoding="utf-8") as f:
        json.dump(r, f, indent=2, ensure_ascii=False)
    pd_pred = partes["test"][["img_id", "patient_id", "diagnostic", "fitspatrick"]].copy()
    for i, c in enumerate(["ACK", "BCC", "MEL", "NEV", "SCC", "SEK"]):
        pd_pred[f"p_{c}"] = p_test[:, i]
    pd_pred.to_csv(f"resultados/predicciones_prueba_{nombre}.csv", index=False)
    return r, p_test

modelo_v2, capa_v2, ruta_v2, hist_v2 = entrenar("MobileNetV2", ds_train, ds_val, Y["train"], "resultados/modelos")
resultados["MobileNetV2"], p_test_v2 = evaluar(modelo_v2, "MobileNetV2", ruta_v2, hist_v2)
print(json.dumps(resultados["MobileNetV2"]["prueba"], indent=2, ensure_ascii=False))

## 6. Figuras para el documento: curvas de entrenamiento y matriz de confusión

In [ ]:
import matplotlib.pyplot as plt
from sklearn.metrics import ConfusionMatrixDisplay

def figuras(nombre, hist):
    h = {k: hist["fase1"].get(k, []) + hist["fase2"].get(k, []) for k in ("auc", "val_auc", "loss", "val_loss")}
    fig, ax = plt.subplots(1, 2, figsize=(10, 3.5))
    ax[0].plot(h["auc"], label="entrenamiento"); ax[0].plot(h["val_auc"], label="validación")
    ax[0].axvline(len(hist["fase1"]["auc"]) - 0.5, ls="--", c="gray"); ax[0].set_title("AUC-ROC"); ax[0].legend()
    ax[1].plot(h["loss"], label="entrenamiento"); ax[1].plot(h["val_loss"], label="validación")
    ax[1].axvline(len(hist["fase1"]["loss"]) - 0.5, ls="--", c="gray"); ax[1].set_title("Pérdida"); ax[1].legend()
    for a in ax: a.set_xlabel("Época")
    fig.tight_layout(); fig.savefig(f"resultados/curvas_{nombre}.png", dpi=200); plt.show()
    mc = np.array(resultados[nombre]["prueba"]["matriz_confusion"]["valores"])
    disp = ConfusionMatrixDisplay(mc, display_labels=["ACK", "BCC", "MEL", "NEV", "SCC", "SEK"])
    disp.plot(cmap="Blues", colorbar=False); plt.title(f"Matriz de confusión – prueba interna ({nombre})")
    plt.tight_layout(); plt.savefig(f"resultados/matriz_confusion_{nombre}.png", dpi=200); plt.show()

figuras("MobileNetV2", hist_v2)

## 7. (Opcional) Validación cruzada de 5 pliegues con MobileNetV2
Tarda unas 5 veces el entrenamiento anterior. Actívala si hay tiempo de GPU (`HACER_VC = True`).

In [ ]:
HACER_VC = False
if HACER_VC:
    from sklearn.model_selection import StratifiedGroupKFold
    dev = df[df["conjunto"] != "test"].reset_index(drop=True)
    X_dev = np.concatenate([X["train"], X["val"]]); Y_dev = np.concatenate([Y["train"], Y["val"]])
    orden = np.concatenate([np.where(df["conjunto"] == "train")[0], np.where(df["conjunto"] == "val")[0]])
    grupos = df.loc[orden, "patient_id"].to_numpy()
    vc = []
    for k, (itr, iva) in enumerate(StratifiedGroupKFold(5, shuffle=True, random_state=42).split(X_dev, Y_dev, grupos)):
        m, _, ruta, _ = entrenar("MobileNetV2", crear_dataset(X_dev[itr], Y_dev[itr], aumentar=True),
                                 crear_dataset(X_dev[iva], Y_dev[iva]), Y_dev[itr], f"resultados/vc/pliegue{k}",
                                 epocas_fase1=10, epocas_fase2=15)
        mk = metricas(Y_dev[iva], m.predict(crear_dataset(X_dev[iva], Y_dev[iva]), verbose=0))
        vc.append({"pliegue": k, "exactitud": mk["exactitud"], "auc_roc_macro": mk["auc_roc_macro"]})
        print(vc[-1])
    resumen_vc = {"pliegues": vc,
                  "exactitud_media": float(np.mean([v["exactitud"] for v in vc])), "exactitud_de": float(np.std([v["exactitud"] for v in vc])),
                  "auc_media": float(np.mean([v["auc_roc_macro"] for v in vc])), "auc_de": float(np.std([v["auc_roc_macro"] for v in vc]))}
    json.dump(resumen_vc, open("resultados/validacion_cruzada_MobileNetV2.json", "w"), indent=2)
    print(resumen_vc)

## 8. MobileNetV3-Large como referencia de comparación (HU16) y regla de la decisión D1

In [ ]:
from training.evaluacion import decidir_arquitectura
modelo_v3, capa_v3, ruta_v3, hist_v3 = entrenar("MobileNetV3Large", ds_train, ds_val, Y["train"], "resultados/modelos")
resultados["MobileNetV3Large"], _ = evaluar(modelo_v3, "MobileNetV3Large", ruta_v3, hist_v3)
figuras("MobileNetV3Large", hist_v3)

import pandas as pd
tabla = pd.DataFrame({n: {
    "Exactitud": r["prueba"]["exactitud"], "Exactitud balanceada": r["prueba"]["exactitud_balanceada"],
    "AUC-ROC macro": r["prueba"]["auc_roc_macro"], "IC 95 %": r["prueba"]["auc_roc_macro_ic95"],
    "AUC-ROC binario maligno": r["prueba"]["auc_roc_binario_maligno"],
    "Sensibilidad MEL": r["prueba"]["sensibilidad_por_clase"]["MEL"],
    "Inferencia CPU (ms)": r["inferencia"]["mediana_ms_cpu"], "Tamaño (MB)": r["inferencia"]["tamano_mb"],
    "Cumple IE4": r["prueba"]["cumple_ie4"]} for n, r in resultados.items()})
display(tabla)
decision = decidir_arquitectura(resultados["MobileNetV2"]["prueba"], resultados["MobileNetV3Large"]["prueba"])
print("Decisión D1:", decision)
json.dump({"tabla": tabla.astype(str).to_dict(), "decision": decision}, open("resultados/comparacion_HU16.json", "w"), indent=2, ensure_ascii=False)

## 9. Ejemplos de Grad-CAM para el documento (HU04)

In [ ]:
from app.gradcam import calcular_gradcam
from app.preprocesamiento import normalizar
from PIL import Image
clases = ["ACK", "BCC", "MEL", "NEV", "SCC", "SEK"]
fig, ax = plt.subplots(2, 6, figsize=(16, 6))
for j, c in enumerate(clases):
    idx = np.where(Y["test"] == j)[0][0]
    x = normalizar(X["test"][idx:idx + 1].astype(np.float32))
    p = modelo_v2.predict(x, verbose=0)[0]
    mapa = calcular_gradcam(modelo_v2, x, int(p.argmax()), capa_v2)
    ax[0, j].imshow(X["test"][idx]); ax[0, j].set_title(f"Real: {c}"); ax[0, j].axis("off")
    ax[1, j].imshow(X["test"][idx]); ax[1, j].imshow(np.array(Image.fromarray(np.uint8(mapa * 255)).resize((224, 224))), cmap="jet", alpha=0.4)
    ax[1, j].set_title(f"Pred.: {clases[p.argmax()]} ({p.max():.2f})"); ax[1, j].axis("off")
plt.tight_layout(); plt.savefig("resultados/gradcam_ejemplos.png", dpi=200); plt.show()

## 10. Calibración del control de calidad (HU09) con las fotos de validación

In [ ]:
import shutil
os.makedirs("/content/fotos_val", exist_ok=True)
for r in partes["val"]["ruta"]:
    shutil.copy(r, "/content/fotos_val/")
!python scripts/calibrar_calidad.py --carpeta /content/fotos_val --salida resultados/calidad_HU09.json --max 300

## 11. Exportar el modelo para la API y descargar los resultados
El umbral de confianza de HU12 se sugiere con el conjunto de **validación** (no con la prueba): el menor umbral con el que la exactitud de los resultados sin advertencia es de al menos 90 %. Debe validarlo el PO.

In [ ]:
import datetime, zipfile
arq_final = "MobileNetV3Large" if decision.startswith("MobileNetV3Large") else "MobileNetV2"
ruta_final = ruta_v3 if arq_final == "MobileNetV3Large" else ruta_v2
capa_final = capa_v3 if arq_final == "MobileNetV3Large" else capa_v2
sugerido = next((f["umbral"] for f in resultados[arq_final]["confianza_validacion"]
                 if f["exactitud_sin_advertencia"] and f["exactitud_sin_advertencia"] >= 0.90), None)
os.makedirs("models", exist_ok=True)
shutil.copy(ruta_final, "models/modelo.keras")
info = {"clases": clases, "arquitectura": arq_final, "capa_gradcam": capa_final,
        "version": f"{arq_final.lower()}-{datetime.date.today():%Y%m%d}",
        "umbral_confianza_sugerido": sugerido,
        "tensorflow": tf.__version__, "keras": keras.__version__,
        "exactitud_prueba": resultados[arq_final]["prueba"]["exactitud"],
        "auc_roc_prueba": resultados[arq_final]["prueba"]["auc_roc_macro"]}
json.dump(info, open("models/modelo_info.json", "w"), indent=2)
print(json.dumps(info, indent=2))
with zipfile.ZipFile("/content/resultados_entrenamiento.zip", "w", zipfile.ZIP_DEFLATED) as z:
    for carpeta in ("resultados", "models"):
        for raiz, _, archivos in os.walk(carpeta):
            for a in archivos:
                if "/vc/" in raiz:
                    continue
                z.write(os.path.join(raiz, a))
from google.colab import files
files.download("/content/resultados_entrenamiento.zip")